In [ ]:
import pandas as pd
import re
import nltk
from sklearn.metrics import confusion_matrix

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix


In [ ]:
nltk.download('stopwords')

In [ ]:
df = pd.read_csv("fake_news.csv")

print(df.head())

# missing values
print(df.isnull().sum())

# Fill missing values
df = df.fillna('')



In [ ]:
# Combine title and text
df['content'] = df['title'] + " " + df['text']


In [ ]:
# text_cleaning
port_stem = PorterStemmer()

def stemming(content):

    # Remove special characters
    content = re.sub('[^a-zA-Z]', ' ', content)

    # Convert to lowercase
    content = content.lower()

    # Split words
    content = content.split()

    # Remove stopwords and apply stemming
    content = [port_stem.stem(word) for word in content
               if not word in stopwords.words('english')]

    # Join words again
    content = ' '.join(content)

    return content



In [ ]:
# Apply cleaning
df['content'] = df['content'].apply(stemming)

# separate features and labels
X = df['content'].values
Y = df['label'].values



In [ ]:
# train test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    stratify=Y,
    random_state=2
)



In [ ]:
# convert text to numbers
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)



In [ ]:
# model training
model = LogisticRegression()

model.fit(X_train, Y_train)

# predict
X_train_prediction = model.predict(X_train)
training_accuracy = accuracy_score(Y_train, X_train_prediction)

print("\nTraining Accuracy:", training_accuracy)




In [ ]:
# Test Accuracy
X_test_prediction = model.predict(X_test)
test_accuracy = accuracy_score(Y_test, X_test_prediction)

print("Test Accuracy:", test_accuracy)




In [ ]:
print("\nClassification Report:\n")
print(classification_report(Y_test, X_test_prediction))

print(confusion_matrix(Y_test, X_test_prediction))



In [ ]:
# predict news
news = input("\nEnter News Article: ")

news = stemming(news)

news_vector = vectorizer.transform([news])

prediction = model.predict(news_vector)

if prediction[0] == 1:
    print("\nThe News is REAL")
else:
    print("\nThe News is FAKE")
